# Verified competition model — CPU check

**Local reference: 329/400 correct (82.25%).** This is the selected verified candidate from the completed experiments. Model identity: `e86b0e136736db50`.

Choose **Runtime → Change runtime type → CPU**, connect, then **Runtime → Run all**. Setup may take several minutes; the 10-second target applies to all 400 predictions after setup. This notebook loads trained weights and does not train a model.

The notebook contains the model, the public demo, and unmodified official evaluator. It installs model packages in a separate folder, checks all embedded file hashes, tests prediction safety and speed, and runs three official evaluation processes. It needs no GitHub login, Google Drive mounting, or paid runtime.

The demo has been reused during development. Its accuracy does not establish hidden-test accuracy or a winning leaderboard position. Colab results will be measured when you run this notebook; the numbers above were measured locally.

Official rules and materials: https://github.com/GDG-UAM/ml-contest-materials


In [ ]:
#@title 1. Prepare isolated model packages
import os,sys,subprocess,json
from pathlib import Path
packages=Path('/content/gdg_packages_e86b0e1367')
packages.mkdir(exist_ok=True)
requirements=['numpy==2.5.4', 'numba==0.68.1']
result=subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-input',
                       '--only-binary=:all:','--target',str(packages),'--upgrade','--ignore-installed',*requirements],
                      text=True,capture_output=True)
if result.returncode:
    print(result.stdout)
    print(result.stderr)
    result.check_returncode()
print('Model packages ready in their separate folder',flush=True)


In [ ]:
#@title 2. Restore the verified model and demo
import base64,hashlib,io,zipfile
workspace=Path('/content/gdg_check_e86b0e1367')
workspace.mkdir(exist_ok=True)
payload=(
'...'
)
manifest={'candidate/core_features.py': 'beb37e3226a6e3b684f96daa46047b49e3df31ca2d4537f03e2748fdc4db1b08', 'candidate/fused_features.py': 'f06df206f727b0ce196a355a6cd35899998655d04b305e0141131ec033384aea', 'candidate/model.py': 'bc5005c69013a3453989501e2333f00d8ec1f8d46c362ba637645df4d555d039', 'candidate/weights.npz': '1bfd5f56935cc7dc30fad631d2b35ba2c15864cde6f5dc2f27af39999dc6855d', 'candidate/requirements.txt': '821a15afd8cb03b7285e7b8b24cb910d96747681fa0d1e0b8bb5aa93db8cb77e', 'harness.py': '2b6c1d7af737273280e3ff83229b7f52a7cf22ae1aac3fe813602e37231d0bec', 'competition_api.py': '3c1ccf44ae644687c4de28949172c94c7cb31ba1ce4b95368fad043acd51b92f', 'data/test_demo.npz': '03fce51b3dacfaaa04418c4907def59874c4b6963c0561c9ccac83bbca90aac0', 'verify_contract.py': 'd4fe6c87ca07e46d886dbf58b66f7eeefdbc2c48af329e58a809521880d87344', 'local_reference.json': '657749512c51e6bccefe8c4ec74b58c707af7c0d7ac5fdf4958fa78b0221fdce'}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    assert set(archive.namelist())==set(manifest)
    for name,digest in manifest.items():
        content=archive.read(name)
        assert hashlib.sha256(content).hexdigest()==digest,name
        destination=workspace/name
        destination.parent.mkdir(parents=True,exist_ok=True)
        destination.write_bytes(content)
print('Verified and restored',len(manifest),'files',flush=True)


In [ ]:
#@title 3. Test all 400 predictions, missing data, and output safety
environment=dict(os.environ,OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',
                 NUMEXPR_NUM_THREADS='1',VECLIB_MAXIMUM_THREADS='1',PYTHONPATH=str(packages))
subprocess.run([sys.executable,'verify_contract.py','--model','candidate/model.py','--report','colab_contract.json'],
               cwd=workspace,env=environment,check=True,timeout=240)


In [ ]:
#@title 4. Run the unmodified official evaluator three times
subprocess.run([sys.executable,'harness.py','evaluate','--model','candidate/model.py',
                '--source','file:data/test_demo.npz','--runs','3','--out','official_results'],
               cwd=workspace,env=environment,check=True,timeout=600)


In [ ]:
#@title 5. Show results and download the model plus evidence
contract=json.loads((workspace/'colab_contract.json').read_text())
runs=[json.loads(p.read_text()) for p in (workspace/'official_results').rglob('run_*.json')]
assert len(runs)==3
assert all(r['status']=='ok' and r['n_scored']==400 and
           sum(r[k] for k in ('n_exception','n_invalid','n_soft_timeout','n_hard_timeout'))==0 for r in runs)
summary=dict(correct=contract['correct'],samples=400,accuracy=contract['accuracy'],
             all_400_prediction_seconds=contract['full_demo_prediction_seconds'],
             median_prediction_seconds=contract['median_full_demo_seconds'],
             under_10_seconds=contract['all_repetitions_under_10_seconds'],
             model_setup_seconds=contract['setup_seconds'],
             official_scores=[r['score'] for r in runs],
             official_setup_seconds=[r['setup_time_s'] for r in runs],official_runs_passed=3,
             zero_exceptions_and_timeouts=True)
(workspace/'COLAB_SUMMARY.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
import shutil
from google.colab import files
shutil.make_archive(str(workspace/'submission'),'zip',workspace/'candidate')
result_zip=shutil.make_archive('/content/BEST_MODEL_COLAB_RESULTS','zip',workspace)
files.download(result_zip)
